<a href="https://colab.research.google.com/github/FeDevPolis/QuantumXSchool/blob/main/Sessions5%266.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-aer matplotlib pylatexenc ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 84.4 MB/s eta 0:00:00


In [4]:
# 1. Instalar y/o importar librerías necesarias
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# -----------------------------------------------------------------
# FUNCIÓN DE CONSTRUCCIÓN DEL CIRCUITO (GROVER PARA 3 BITS)
# -----------------------------------------------------------------
def create_grover_circuit(target_pattern):
    """
    Construye el circuito de Grover para 3 cúbits según el patrón deseado.
    Aplica 2 iteraciones de Grover para maximizar la probabilidad del estado objetivo.
    """
    # 3 cúbits para los datos + 1 cúbit ancilla para phase kickback
    qc = QuantumCircuit(4, 3)

    # Paso 1: Preparación del cúbit ancilla en el estado |-⟩
    qc.x(3)
    qc.h(3)

    # Paso 2: Crear superposición igual en los cúbits de datos
    qc.h(range(3))
    qc.barrier()

    # Se realizan 2 iteraciones de Grover (~π/4 * √8 ≈ 2)
    for _ in range(2):
        # ---------------------------------------------------------
        # ORÁCULO DINÁMICO (Marca de Fase)
        # ---------------------------------------------------------
        # Aplicar X-gates a las posiciones donde el bit del objetivo es '0'
        for i, bit in enumerate(reversed(target_pattern)):
            if bit == '0':
                qc.x(i)

        # Multi-Control Toffoli para invertir la fase del estado objetivo mediante el ancilla
        qc.mcx([0, 1, 2], 3)

        # Deshacer las X-gates (Uncomputing)
        for i, bit in enumerate(reversed(target_pattern)):
            if bit == '0':
                qc.x(i)

        qc.barrier()

        # ---------------------------------------------------------
        # OPERADOR DE DIFUSIÓN (Amplificación de Amplitud)
        # ---------------------------------------------------------
        qc.h(range(3))
        qc.x(range(3))

        # Multi-controlled Z con el cúbit ancilla
        qc.mcx([0, 1, 2], 3)

        qc.x(range(3))
        qc.h(range(3))
        qc.barrier()

    # Paso final: Medición de los 3 cúbits de datos
    qc.measure(range(3), range(3))
    return qc


# -----------------------------------------------------------------
# FUNCIÓN INTERACTIVA DE EJECUCIÓN Y VISUALIZACIÓN
# -----------------------------------------------------------------
def run_quantum_pattern_search(target):
    # Generar circuito
    qc = create_grover_circuit(target)

    # 1. Mostrar diagrama del circuito cuántico (estilo mpl)
    display(Markdown(f"### Complete Grover Circuit (Target = {target}):"))
    fig_circuit = qc.draw(output="mpl", fold=-1)
    display(fig_circuit)
    plt.close(fig_circuit)

    # 2. Ejecutar simulación con AerSimulator
    simulator = AerSimulator()
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=1024).result()
    counts = result.get_counts()

    display(Markdown("### Run the circuit using AerSimulator"))
    print("Measurement Results:")
    print(counts)

    # 3. Mostrar Histograma de Mediciones
    display(Markdown("### Display the measurement histogram"))
    fig_hist = plot_histogram(
        counts,
        sort='value_desc',
        title=f"Grover Search Results - Target = {target}"
    )
    display(fig_hist)
    plt.close(fig_hist)

    # 4. Reporte del resultado identificado
    dominant_pattern = max(counts, key=counts.get)
    display(Markdown(f"**Pattern identified by the quantum algorithm:** `{dominant_pattern}`"))


# -----------------------------------------------------------------
# MENÚ INTERACTIVO (Entrada de Usuario)
# -----------------------------------------------------------------
widgets.interact(
    run_quantum_pattern_search,
    target=["000", "001", "010", "011", "100", "101", "110", "111"]
);

interactive(children=(Dropdown(description='target', options=('000', '001', '010', '011', '100', '101', '110',…

## Explanation

* **Superposition:** Hadamard ($H$) gates place the three search qubits into an equal superposition state, enabling the quantum computer to evaluate all 8 binary patterns ($000$ to $111$) simultaneously.
* **Oracle & Phase Marking:** The dynamic oracle identifies the target pattern by selectively applying Pauli-$X$ gates to match bits with `'0'`. It uses phase kickback via an ancilla qubit in state $\vert{}-\rangle$ to invert the phase ($\pi$-phase shift) of only the target state, marking it negative without changing state probabilities.
* **Interference & Amplitude Amplification:** The diffusion operator computes the average probability amplitude across all computational states and reflects each state's amplitude about this mean. Constructive interference amplifies the probability amplitude of the phase-marked target state while destructive interference suppresses the remaining non-target states.
* **Pattern Identification:** Upon measurement, quantum wave-function collapse forces the system into the state with the highest probability, allowing `AerSimulator` to output the target pattern as the identified result after applying two Grover iterations.


## Test the dynamic oracle

Select different inputs directly using the interactive dropdown menu:

| Target Pattern | Expected Dominant Result |
| :---: | :---: |
| **101** | **101** |
| **000** | **000** |
| **011** | **011** |
| **110** | **110** |

Changing the selected option in the interactive menu automatically updates the marked state, the quantum circuit, and the dominant measurement result in real time. This demonstrates that the oracle is dynamically constructed rather than hard-coded for a single target.

## Conclusion

Grover's algorithm was implemented using Qiskit to search an eight-state space for a user-selected 3-bit pattern. The experiment demonstrates how superposition, phase marking, interference, and amplitude amplification work together to amplify the target state's probability. The dynamic oracle construction allows the application to search for any 3-bit pattern without altering the underlying circuit code.